# Misinformation Models—
Trains all models to date and saves them.

Claude helped on implementation details. Logic and concepts are human generated.

### Import configuration and packages ###

In [1]:
##TO RUN ON COLAB ONLY


#installs
!pip install tweet-preprocessor==0.5.0 feedparser whoosh iterative-stratification fastapi uvicorn
!pip install nltk spacy

#imports
import sys
from pathlib import Path
from google.colab import drive
import pandas as pd
import numpy as np
import torch

#mount google drive and set path-related variables.
drive.mount('/content/drive')
BASE_DIR=Path("/content/drive/MyDrive/linguistic_markers")
SPRINT_DIR = BASE_DIR / "581_Sprint_4"
SRC_DIR = SPRINT_DIR / "src"
DATA_DIR = BASE_DIR / "data" / "final_splits"

# Add the SPRINT_DIR to the system path so Python can find modules like 'cnn_baseline'
sys.path.insert(0, str(Path.cwd()))
sys.path.insert(1, str(SPRINT_DIR))
sys.path.insert(2, str(SRC_DIR))
sys.path.insert(3, str(DATA_DIR))
sys.path.insert(4, str(BASE_DIR))

# Define the device for training
DEVICE = torch.device("cuda" + ":0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


import cnn_baseline as cnn
#Running this will import FastText vector file, stored on HuggingFace and is >4gb.
from config import FASTTEXT_PATH, TARGETS, SEED
from preprocess import preprocess
from metrics import compute_metrics, print_confusion_matrix, print_sklearn_report, error_analysis, print_report
import random


random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)


  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 468.8/468.8 kB 15.3 MB/s eta 0:00:00
  Created wheel for tweet-preprocessor: filename=tweet_preprocessor-0.5.0-py3-none-any.whl size=7928 sha256=d6fd2a83ece5f01ba167ec0f4b03a5093d4c03b06764a1d9b2b1b6d3d4f86fcf
  Stored in directory: /root/.cache/pip/wheels/4b/6e/04/d26d41ed041dd0318b112367564452d05a4835d1a3f8e37518
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6046 sha256=df36c5b3a2584144ee066caa34283d7978522f1f700afcb47e2c1adf5964c51f
  Stored in directory: /root/.cache/pip/wheels/03/f5/1a/23761066dac1d0e8e683e5fdb27e12de53209d05a4a37e6246
Successfully built tweet-preprocessor sgmllib3k
Mounted at /content/drive
Using device: cuda:0


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


cc.en.300.vec:   0%|          | 0.00/4.51G [00:00<?, ?B/s]

In [2]:
#DO NOT RUN IF USING COLAB


#import sys
#from pathlib import Path
#sys.path.insert(0, str(Path.cwd()))

#import pandas as pd
#import numpy as np
#import torch                       # CNN only
#import cnn_baseline as cnn         # CNN only

# Running this will import FastText vector file, stored on HuggingFace and is >4gb.
#from config import DATA_DIR, FASTTEXT_PATH, TARGETS

#from preprocess import preprocess  # CNN only
#from metrics import compute_metrics, print_confusion_matrix, print_sklearn_report, error_analysis, print_report

#DEVICE = torch.device(             # CNN only
 #   "mps"  if torch.backends.mps.is_available()  else
  #  "cuda" if torch.cuda.is_available()           else
   # "cpu"
#)
#print(f"Device: {DEVICE}")         # CNN only
#print(f"Targets: {TARGETS}")

In [3]:

# Load data (shared across all models)
train_rows = cnn.load_csv(DATA_DIR / "mis_df_train.csv")
dev_rows   = cnn.load_csv(DATA_DIR / "mis_df_dev.csv")
print(f"Train: {len(train_rows)} | Dev: {len(dev_rows)}")

Train: 600 | Dev: 200


In [4]:
import pathlib
import sys

# Manually add Path to the builtins or the sys module
# so the script can see it when it loads
import builtins
builtins.Path = pathlib.Path

import cnn_mtl_no_ling as cnn_mtl_pos

## Run models


In [5]:
# Each model should produce (preds, labels, probs) and store them in `results`.
# Add new models below following the same pattern.
results = {}

#added metrics_cache for later use in ordering all models by Macro F1 and making easy displays
metrics_cache = {}


In [6]:
import joblib, pickle, torch.nn as nn
from pathlib import Path

SAVE_DIR = SPRINT_DIR / "saved_models"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

def save_checkpoint(key, model=None, result=None):
    """Save model weights and/or (preds, labels, probs) result tuple."""
    safe_key = key.replace(" ", "_").replace("/", "-").replace("(", "").replace(")", "").replace("+", "plus")
    saved = []

    if model is not None:
        if isinstance(model, nn.Module):
            torch.save(model.state_dict(), SAVE_DIR / f"{safe_key}.pt")
            saved.append("state_dict")
        else:
            joblib.dump(model, SAVE_DIR / f"{safe_key}.joblib")
            saved.append("joblib")

    if result is not None:
        with open(SAVE_DIR / f"{safe_key}_result.pkl", "wb") as f:
            pickle.dump(result, f)
        saved.append("result")

    if saved:
        print(f"  [saved] {safe_key} ({', '.join(saved)})")

In [7]:
# Model name constants — define once, use everywhere
M_TEXT_CNN                 = "TextCNN"
M_TEXT_CNN_TRANSFER        = "TextCNN Transfer"
M_TEXT_CNN_TRANSFER_SHARED = "TextCNN_Transfer_shared"
M_CNN_MTL_LING             = "TextCNN MTL (POS + linguistic)"
M_CNN_MTL_POS              = "TextCNN MTL (POS)"
M_LOGREG                   = "LogReg"
M_LOGREG_EMBED             = "LogReg (+ embeddings)"
M_LOGREG_MTL_POS           = "LogReg MTL (cascaded POS)"

M_ENSEMBLE_SOFT_VOTE       = "Soft Vote Ensemble"
M_ENSEMBLE_MOTIVATED       = "Motivated Ensemble"
M_ENSEMBLE_LEAN_SOFT_VOTE  = "Lean Soft Vote"
M_ENSEMBLE_LEAN_MOTIVATED  = "Lean Motivated Ensemble"

CNN_MODELS      = [M_TEXT_CNN, M_TEXT_CNN_TRANSFER, M_CNN_MTL_LING, M_CNN_MTL_POS]
LR_MODELS       = [M_LOGREG, M_LOGREG_EMBED, M_LOGREG_MTL_POS]
BASE_MODELS     = LR_MODELS + CNN_MODELS
ENSEMBLE_MODELS = [M_ENSEMBLE_SOFT_VOTE, M_ENSEMBLE_MOTIVATED,
                   M_ENSEMBLE_LEAN_SOFT_VOTE, M_ENSEMBLE_LEAN_MOTIVATED]


def rank_key(metrics_key):
    """Sort key for model selection: primary = Macro F1, tiebreaker = AUC-ROC."""
    m = metrics_cache[metrics_key]
    return (m["macro_f1"], m.get("auc_roc", 0.0))


def display_metrics(key_prefix):
    """Display a metrics table for one ensemble/model prefix across all TARGETS."""
    pd.set_option("display.float_format", "{:.4f}".format)
    rows = []
    for t in TARGETS:
        key = f"{key_prefix} — {t}"
        m = metrics_cache[key]
        rows.append({
            "Model": key,
            "Macro F1": m["macro_f1"],
            "F1 (not-op)": m["f1_class0"],
            "F1 (opinion)": m["f1_class1"],
            "F1.5 (recall-weighted)": m["fbeta_class1"],
            "AUC-ROC": m.get("auc_roc", float("nan")),
        })
    return pd.DataFrame(rows).set_index("Model")

#### CNN Baseline

In [8]:
#CNN Baseline
vocab        = cnn.build_vocab(train_rows, preprocess)
embed_matrix = cnn.load_fasttext_vectors(FASTTEXT_PATH, vocab)
train_loader = cnn.make_loader(train_rows, vocab, shuffle=True,  tokenize_fn=preprocess)
dev_loader   = cnn.make_loader(dev_rows,   vocab, shuffle=False, tokenize_fn=preprocess)

for target in TARGETS:
    model = cnn.TextCNN(len(vocab), embed_matrix).to(DEVICE)
    model = cnn.train_model(model, train_loader, dev_loader, train_rows, DEVICE,
                            train_targets=[target])
    key = f"{M_TEXT_CNN} — {target}"
    results[key] = cnn.predict(model, dev_loader, DEVICE, target=target)
    metrics_cache[key] = compute_metrics(*results[key])
    save_checkpoint(key, model=model, result=results[key])

Loading FastText vectors from /root/.cache/huggingface/hub/datasets--COLX523--fasttext-cc-en-300/snapshots/63ba06b23eb770a6fe0f6c17b0de981e43aec6aa/cc.en.300.vec …
  3907/4088 vocab tokens found in FastText vectors (95.6%)
Epoch   1 | loss=0.7675 | avg_dev_f1=0.6700 (opinion_label: 0.6700)
Epoch   2 | loss=0.6532 | avg_dev_f1=0.6862 (opinion_label: 0.6862)
Epoch   3 | loss=0.6069 | avg_dev_f1=0.7238 (opinion_label: 0.7238)
Epoch   4 | loss=0.5540 | avg_dev_f1=0.7234 (opinion_label: 0.7234)
Epoch   5 | loss=0.4960 | avg_dev_f1=0.6800 (opinion_label: 0.6800)
Epoch   6 | loss=0.4604 | avg_dev_f1=0.7287 (opinion_label: 0.7287)
Epoch   7 | loss=0.3911 | avg_dev_f1=0.7196 (opinion_label: 0.7196)
Epoch   8 | loss=0.3478 | avg_dev_f1=0.7242 (opinion_label: 0.7242)
Epoch   9 | loss=0.3092 | avg_dev_f1=0.7410 (opinion_label: 0.7410)
Epoch  10 | loss=0.2668 | avg_dev_f1=0.7147 (opinion_label: 0.7147)
Epoch  11 | loss=0.2322 | avg_dev_f1=0.7339 (opinion_label: 0.7339)
Epoch  12 | loss=0.2030 | avg

#### CNN Transfer Learning

In [9]:
# CNN Transfer Learning
cnn_model = cnn.TextCNN(len(vocab), embed_matrix).to(DEVICE)
cnn_model = cnn.train_model(cnn_model, train_loader, dev_loader, train_rows, DEVICE)
save_checkpoint(M_TEXT_CNN_TRANSFER_SHARED, model=cnn_model)

for target in TARGETS:
    key = f"{M_TEXT_CNN_TRANSFER} — {target}"
    results[key] = cnn.predict(cnn_model, dev_loader, DEVICE, target=target)
    metrics_cache[key] = compute_metrics(*results[key])
    save_checkpoint(key, result=results[key])

Epoch   1 | loss=1.7590 | avg_dev_f1=0.6967 (opinion_label: 0.7076 | misinformation_label: 0.6858)
Epoch   2 | loss=1.5097 | avg_dev_f1=0.6910 (opinion_label: 0.6922 | misinformation_label: 0.6898)
Epoch   3 | loss=1.3458 | avg_dev_f1=0.7262 (opinion_label: 0.6889 | misinformation_label: 0.7635)
Epoch   4 | loss=1.1901 | avg_dev_f1=0.7498 (opinion_label: 0.7177 | misinformation_label: 0.7818)
Epoch   5 | loss=1.0373 | avg_dev_f1=0.7580 (opinion_label: 0.7033 | misinformation_label: 0.8128)
Epoch   6 | loss=0.9232 | avg_dev_f1=0.7640 (opinion_label: 0.7076 | misinformation_label: 0.8203)
Epoch   7 | loss=0.7993 | avg_dev_f1=0.7687 (opinion_label: 0.7129 | misinformation_label: 0.8245)
Epoch   8 | loss=0.7516 | avg_dev_f1=0.7708 (opinion_label: 0.7134 | misinformation_label: 0.8282)
Epoch   9 | loss=0.6607 | avg_dev_f1=0.7872 (opinion_label: 0.7134 | misinformation_label: 0.8609)
Epoch  10 | loss=0.5938 | avg_dev_f1=0.7769 (opinion_label: 0.7023 | misinformation_label: 0.8515)
Epoch  11 

#### CNN MTL — POS & Linguistic Features


In [10]:
import cnn_mtl_ling as cnn_mtl_ling

train_loader_ling = cnn_mtl_ling.make_loader(train_rows, vocab, shuffle=True,  tokenize_fn=preprocess)
dev_loader_ling   = cnn_mtl_ling.make_loader(dev_rows,   vocab, shuffle=False, tokenize_fn=preprocess)

for target in TARGETS:
    model = cnn_mtl_ling.TextCNN(len(vocab), embed_matrix).to(DEVICE)
    model = cnn_mtl_ling.train_model(model, train_loader_ling, dev_loader_ling, train_rows, DEVICE,
                                     train_targets=[target])
    key = f"{M_CNN_MTL_LING} — {target}"
    results[key] = cnn_mtl_ling.predict(model, dev_loader_ling, DEVICE, target=target)
    metrics_cache[key] = compute_metrics(*results[key])
    save_checkpoint(key, model=model, result=results[key])

Epoch   1 | loss=0.7583 | avg_dev_f1=0.6987 (opinion_label: 0.6987)
Epoch   2 | loss=0.6579 | avg_dev_f1=0.6922 (opinion_label: 0.6922)
Epoch   3 | loss=0.6055 | avg_dev_f1=0.6648 (opinion_label: 0.6648)
Epoch   4 | loss=0.5716 | avg_dev_f1=0.6944 (opinion_label: 0.6944)
Epoch   5 | loss=0.5110 | avg_dev_f1=0.6750 (opinion_label: 0.6750)
Epoch   6 | loss=0.4479 | avg_dev_f1=0.7374 (opinion_label: 0.7374)
Epoch   7 | loss=0.4070 | avg_dev_f1=0.6946 (opinion_label: 0.6946)
Epoch   8 | loss=0.3581 | avg_dev_f1=0.7230 (opinion_label: 0.7230)
Epoch   9 | loss=0.2974 | avg_dev_f1=0.7190 (opinion_label: 0.7190)
Epoch  10 | loss=0.2555 | avg_dev_f1=0.7193 (opinion_label: 0.7193)
Epoch  11 | loss=0.2285 | avg_dev_f1=0.7190 (opinion_label: 0.7190)
  Early stopping (no improvement for 5 epochs). Best F1: 0.7374
  [saved] TextCNN_MTL_POS_plus_linguistic_—_opinion_label (state_dict, result)
Epoch   1 | loss=0.9857 | avg_dev_f1=0.5830 (misinformation_label: 0.5830)
Epoch   2 | loss=0.8269 | avg_dev_

#### CNN MTL - POS Features only


In [11]:
import cnn_mtl_no_ling as cnn_mtl_pos

train_loader_pos = cnn_mtl_pos.make_loader(train_rows, vocab, shuffle=True,  tokenize_fn=preprocess)
dev_loader_pos   = cnn_mtl_pos.make_loader(dev_rows,   vocab, shuffle=False, tokenize_fn=preprocess)

for target in TARGETS:
    model = cnn_mtl_pos.TextCNN(len(vocab), embed_matrix).to(DEVICE)
    model = cnn_mtl_pos.train_model(model, train_loader_pos, dev_loader_pos, train_rows, DEVICE,
                                    train_targets=[target])
    key = f"{M_CNN_MTL_POS} — {target}"
    results[key] = cnn_mtl_pos.predict(model, dev_loader_pos, DEVICE, target=target)
    metrics_cache[key] = compute_metrics(*results[key])
    save_checkpoint(key, model=model, result=results[key])

Epoch   1 | loss=0.7599 | avg_dev_f1=0.6792 (opinion_label: 0.6792)
Epoch   2 | loss=0.6590 | avg_dev_f1=0.6928 (opinion_label: 0.6928)
Epoch   3 | loss=0.6039 | avg_dev_f1=0.7046 (opinion_label: 0.7046)
Epoch   4 | loss=0.5529 | avg_dev_f1=0.7037 (opinion_label: 0.7037)
Epoch   5 | loss=0.5071 | avg_dev_f1=0.7199 (opinion_label: 0.7199)
Epoch   6 | loss=0.4477 | avg_dev_f1=0.7081 (opinion_label: 0.7081)
Epoch   7 | loss=0.4074 | avg_dev_f1=0.7368 (opinion_label: 0.7368)
Epoch   8 | loss=0.3524 | avg_dev_f1=0.7383 (opinion_label: 0.7383)
Epoch   9 | loss=0.2995 | avg_dev_f1=0.7193 (opinion_label: 0.7193)
Epoch  10 | loss=0.2615 | avg_dev_f1=0.7246 (opinion_label: 0.7246)
Epoch  11 | loss=0.2199 | avg_dev_f1=0.7261 (opinion_label: 0.7261)
Epoch  12 | loss=0.2065 | avg_dev_f1=0.7368 (opinion_label: 0.7368)
Epoch  13 | loss=0.1686 | avg_dev_f1=0.7024 (opinion_label: 0.7024)
  Early stopping (no improvement for 5 epochs). Best F1: 0.7383
  [saved] TextCNN_MTL_POS_—_opinion_label (state_dic

#### Logistic Regression Baseline

In [12]:
import logreg_baseline as lr

for target in TARGETS:
    key = f"{M_LOGREG} — {target}"
    results[key] = lr.run(train_rows, dev_rows, task=target)
    metrics_cache[key] = compute_metrics(*results[key])
    save_checkpoint(key, result=results[key])


── Logistic Regression  [opinion_label] ──
  Building features (TF-IDF + linguistic)
  Feature matrix: train=(600, 2504), dev=(200, 2504)
  Running GridSearchCV over C
  Best C: 1.0  |  CV macro-F1: 0.7013
  [saved] LogReg_—_opinion_label (result)

── Logistic Regression  [misinformation_label] ──
  Building features (TF-IDF + linguistic)
  Feature matrix: train=(600, 2504), dev=(200, 2504)
  Running GridSearchCV over C
  Best C: 1.0  |  CV macro-F1: 0.8514
  [saved] LogReg_—_misinformation_label (result)


#### Logistic Regression Transfer Learning

In [13]:
import logreg_transfer as lr_transfer

for target in TARGETS:
    key = f"{M_LOGREG_EMBED} — {target}"
    results[key] = lr_transfer.run(train_rows, dev_rows, task=target)
    metrics_cache[key] = compute_metrics(*results[key])
    save_checkpoint(key, result=results[key])


── Logistic Regression  [opinion_label] ──
  Building features (TF-IDF, linguistic features, FastText embeddings)
Loading FastText vectors from /root/.cache/huggingface/hub/datasets--COLX523--fasttext-cc-en-300/snapshots/63ba06b23eb770a6fe0f6c17b0de981e43aec6aa/cc.en.300.vec …
  3566/4375 vocab tokens found in FastText vectors (81.5%)
  Feature matrix: train=(600, 2804), dev=(200, 2804)
  Running GridSearchCV over C
  Best C: 1.0  |  CV macro-F1: 0.7285
  [saved] LogReg_plus_embeddings_—_opinion_label (result)

── Logistic Regression  [misinformation_label] ──
  Building features (TF-IDF, linguistic features, FastText embeddings)
Loading FastText vectors from /root/.cache/huggingface/hub/datasets--COLX523--fasttext-cc-en-300/snapshots/63ba06b23eb770a6fe0f6c17b0de981e43aec6aa/cc.en.300.vec …
  3566/4375 vocab tokens found in FastText vectors (81.5%)
  Feature matrix: train=(600, 2804), dev=(200, 2804)
  Running GridSearchCV over C
  Best C: 1.0  |  CV macro-F1: 0.8716
  [saved] LogReg_

#### LogReg MTL — Cascaded POS Prediction (Sprint 3)


In [14]:
import logreg_mtl as lr_mtl

for target in TARGETS:
    key = f"{M_LOGREG_MTL_POS} — {target}"
    results[key] = lr_mtl.run(train_rows, dev_rows, task=target)
    metrics_cache[key] = compute_metrics(*results[key])
    save_checkpoint(key, result=results[key])


── LogReg MTL Cascaded POS  [opinion_label] ──
  Building features (TF-IDF, FastText, linguistic, cascaded POS)
Loading FastText vectors from /root/.cache/huggingface/hub/datasets--COLX523--fasttext-cc-en-300/snapshots/63ba06b23eb770a6fe0f6c17b0de981e43aec6aa/cc.en.300.vec …
  3566/4375 vocab tokens found in FastText vectors (81.5%)
  Running secondary task: POS tagging 600 documents …
  POS distribution feature dim: 15 tags
  Running secondary task: POS tagging 200 documents …
  POS distribution feature dim: 15 tags
  Feature matrix: train=(600, 2819), dev=(200, 2819)
  Running GridSearchCV over C
  Best C: 0.1  |  CV macro-F1: 0.7188
  [saved] LogReg_MTL_cascaded_POS_—_opinion_label (result)

── LogReg MTL Cascaded POS  [misinformation_label] ──
  Building features (TF-IDF, FastText, linguistic, cascaded POS)
Loading FastText vectors from /root/.cache/huggingface/hub/datasets--COLX523--fasttext-cc-en-300/snapshots/63ba06b23eb770a6fe0f6c17b0de981e43aec6aa/cc.en.300.vec …
  3566/4375 

### Simple Ensembling

In [15]:
# Soft vote ensemble
import importlib
import simple_ensemble
importlib.reload(simple_ensemble)
from simple_ensemble import soft_vote

for task in TARGETS:
    preds, labels, probs = soft_vote([results[f"{m} — {task}"] for m in BASE_MODELS])
    key = f"{M_ENSEMBLE_SOFT_VOTE} — {task}"
    results[key] = (preds, labels, probs)
    metrics_cache[key] = compute_metrics(preds, labels, probs)
    save_checkpoint(key, result=results[key])

display_metrics(M_ENSEMBLE_SOFT_VOTE)

  [saved] Soft_Vote_Ensemble_—_opinion_label (result)
  [saved] Soft_Vote_Ensemble_—_misinformation_label (result)


,Macro F1,F1 (not-op),F1 (opinion),F1.5 (recall-weighted),AUC-ROC
Model,,,,,
Soft Vote Ensemble — opinion_label,0.7457,0.7788,0.7126,0.7255,0.7943
Soft Vote Ensemble — misinformation_label,0.9090,0.9527,0.8654,0.8654,0.9669


### Motivated Ensembling

In [16]:
# Motivated (F1-weighted) ensemble
import importlib
import motivated_ensemble
importlib.reload(motivated_ensemble)
from motivated_ensemble import motivated_soft_vote

for task in TARGETS:
    f1_weights = [metrics_cache[f"{m} — {task}"]["macro_f1"] for m in BASE_MODELS]
    preds, labels, probs = motivated_soft_vote(
        model_outputs=[results[f"{m} — {task}"] for m in BASE_MODELS],
        f1_weights=f1_weights,
    )
    key = f"{M_ENSEMBLE_MOTIVATED} — {task}"
    results[key] = (preds, labels, probs)
    metrics_cache[key] = compute_metrics(preds, labels, probs)
    save_checkpoint(key, result=results[key])

display_metrics(M_ENSEMBLE_MOTIVATED)


  Threshold sweep (Macro F1 criterion):
   Threshold    Macro F1   F1 (cl.0)   F1 (cl.1)    Accuracy
  ----------------------------------------------------------
        0.30      0.6525      0.6230      0.6820      0.6550
        0.35      0.6740      0.6561      0.6919      0.6750
        0.40      0.7150      0.7164      0.7136      0.7150
        0.45      0.7345      0.7464      0.7225      0.7350
        0.50      0.7457      0.7788      0.7126      0.7500 ◄
        0.55      0.7379      0.7811      0.6946      0.7450
        0.60      0.7044      0.7724      0.6364      0.7200
        0.65      0.6917      0.7765      0.6069      0.7150
        0.70      0.6452      0.7640      0.5263      0.6850

  Selected threshold: 0.50
  [saved] Motivated_Ensemble_—_opinion_label (result)

  Threshold sweep (Macro F1 criterion):
   Threshold    Macro F1   F1 (cl.0)   F1 (cl.1)    Accuracy
  ----------------------------------------------------------
        0.30      0.9019      0.9441     

,Macro F1,F1 (not-op),F1 (opinion),F1.5 (recall-weighted),AUC-ROC
Model,,,,,
Motivated Ensemble — opinion_label,0.7457,0.7788,0.7126,0.7255,0.7946
Motivated Ensemble — misinformation_label,0.9090,0.9527,0.8654,0.8654,0.9670


#### Lean Ensemble Soft Vote: uses best logreg and best CNN model only (Sprint 3)

In [17]:
# Lean soft vote: best LogReg + best CNN per task (selected dynamically from metrics_cache)
best_cnn = {
    task: max(CNN_MODELS, key=lambda m: rank_key(f"{m} — {task}"))
    for task in TARGETS
}
best_logreg = {
    task: max(LR_MODELS, key=lambda m: rank_key(f"{m} — {task}"))
    for task in TARGETS
}

for task in TARGETS:
    print(f"{task}: best LogReg = {best_logreg[task]} | best CNN = {best_cnn[task]}")

for task in TARGETS:
    preds, labels, probs = soft_vote([
        results[f"{best_logreg[task]} — {task}"],
        results[f"{best_cnn[task]} — {task}"],
    ])
    key = f"{M_ENSEMBLE_LEAN_SOFT_VOTE} — {task}"
    results[key] = (preds, labels, probs)
    metrics_cache[key] = compute_metrics(preds, labels, probs)
    save_checkpoint(key, result=results[key])

display_metrics(M_ENSEMBLE_LEAN_SOFT_VOTE)

opinion_label: best LogReg = LogReg (+ embeddings) | best CNN = TextCNN
misinformation_label: best LogReg = LogReg (+ embeddings) | best CNN = TextCNN MTL (POS + linguistic)
  [saved] Lean_Soft_Vote_—_opinion_label (result)
  [saved] Lean_Soft_Vote_—_misinformation_label (result)


,Macro F1,F1 (not-op),F1 (opinion),F1.5 (recall-weighted),AUC-ROC
Model,,,,,
Lean Soft Vote — opinion_label,0.7613,0.7911,0.7314,0.7462,0.8027
Lean Soft Vote — misinformation_label,0.9090,0.9527,0.8654,0.8654,0.9644


#### Lean Motivated Ensemble: uses best CNN and best logreg models only (Sprint 3)

In [18]:
# Lean motivated ensemble: same 2 models, F1-weighted with threshold sweep
for task in TARGETS:
    lean_models = [best_logreg[task], best_cnn[task]]
    f1_weights = [metrics_cache[f"{m} — {task}"]["macro_f1"] for m in lean_models]
    preds, labels, probs = motivated_soft_vote(
        model_outputs=[results[f"{m} — {task}"] for m in lean_models],
        f1_weights=f1_weights,
    )
    key = f"{M_ENSEMBLE_LEAN_MOTIVATED} — {task}"
    results[key] = (preds, labels, probs)
    metrics_cache[key] = compute_metrics(preds, labels, probs)
    save_checkpoint(key, result=results[key])

display_metrics(M_ENSEMBLE_LEAN_MOTIVATED)


  Threshold sweep (Macro F1 criterion):
   Threshold    Macro F1   F1 (cl.0)   F1 (cl.1)    Accuracy
  ----------------------------------------------------------
        0.30      0.6784      0.6559      0.7009      0.6800
        0.35      0.6995      0.6875      0.7115      0.7000
        0.40      0.7300      0.7327      0.7273      0.7300
        0.45      0.7391      0.7547      0.7234      0.7400
        0.50      0.7613      0.7911      0.7314      0.7650 ◄
        0.55      0.7141      0.7699      0.6584      0.7250
        0.60      0.6968      0.7711      0.6225      0.7150
        0.65      0.6962      0.7812      0.6111      0.7200
        0.70      0.6564      0.7715      0.5414      0.6950

  Selected threshold: 0.50
  [saved] Lean_Motivated_Ensemble_—_opinion_label (result)

  Threshold sweep (Macro F1 criterion):
   Threshold    Macro F1   F1 (cl.0)   F1 (cl.1)    Accuracy
  ----------------------------------------------------------
        0.30      0.8874      0.9319

,Macro F1,F1 (not-op),F1 (opinion),F1.5 (recall-weighted),AUC-ROC
Model,,,,,
Lean Motivated Ensemble — opinion_label,0.7613,0.7911,0.7314,0.7462,0.8031
Lean Motivated Ensemble — misinformation_label,0.9180,0.9553,0.8807,0.8966,0.9644


### All Results by Target

In [19]:
# All results by target — everything is now in metrics_cache
all_rows = [{"Model": name, "Macro F1": m["macro_f1"],
             "F1 (not-op)": m["f1_class0"], "F1 (opinion)": m["f1_class1"],
             "F1.5 (recall-weighted)": m["fbeta_class1"],
             "AUC-ROC": m.get("auc_roc", float("nan"))}
            for name, m in metrics_cache.items()]

all_df = pd.DataFrame(all_rows)

for task, label in [("Opinion", "opinion_label"), ("Misinformation", "misinformation_label")]:
    mask = all_df["Model"].str.endswith(f"— {label}")
    df = (all_df[mask]
          .copy()
          .assign(Model=lambda d: d["Model"].str.replace(f" — {label}", "", regex=False))
          .set_index("Model")
          .sort_values("Macro F1", ascending=False))
    print(task, "Ordered by Macro F1 (Desc)")
    print("─" * 60)
    display(df)
    print()

Opinion Ordered by Macro F1 (Desc)
────────────────────────────────────────────────────────────


,Macro F1,F1 (not-op),F1 (opinion),F1.5 (recall-weighted),AUC-ROC
Model,,,,,
Lean Soft Vote,0.7613,0.7911,0.7314,0.7462,0.8027
Lean Motivated Ensemble,0.7613,0.7911,0.7314,0.7462,0.8031
Soft Vote Ensemble,0.7457,0.7788,0.7126,0.7255,0.7943
Motivated Ensemble,0.7457,0.7788,0.7126,0.7255,0.7946
TextCNN,0.7410,0.7733,0.7086,0.7229,0.7992
TextCNN MTL (POS),0.7383,0.7593,0.7174,0.7454,0.7852
TextCNN MTL (POS + linguistic),0.7374,0.7636,0.7111,0.7330,0.7788
TextCNN Transfer,0.7314,0.7623,0.7006,0.7177,0.7827
LogReg (+ embeddings),0.7023,0.7306,0.6740,0.6962,0.7698



Misinformation Ordered by Macro F1 (Desc)
────────────────────────────────────────────────────────────


,Macro F1,F1 (not-op),F1 (opinion),F1.5 (recall-weighted),AUC-ROC
Model,,,,,
Lean Motivated Ensemble,0.9180,0.9553,0.8807,0.8966,0.9644
Lean Soft Vote,0.9090,0.9527,0.8654,0.8654,0.9644
Soft Vote Ensemble,0.9090,0.9527,0.8654,0.8654,0.9669
Motivated Ensemble,0.9090,0.9527,0.8654,0.8654,0.9670
LogReg (+ embeddings),0.8889,0.9428,0.8350,0.8318,0.9501
TextCNN MTL (POS + linguistic),0.8845,0.9388,0.8302,0.8363,0.9553
LogReg MTL (cascaded POS),0.8831,0.9392,0.8269,0.8269,0.9157
TextCNN MTL (POS),0.8816,0.9396,0.8235,0.8174,0.9587
TextCNN,0.8816,0.9396,0.8235,0.8174,0.9591


### Best Model Analysis: Confusion Matrix & Examples

In [20]:
def print_quadrant_examples(dev_rows, preds, labels, n=3):
    """Print up to n examples from each confusion matrix quadrant."""
    quadrants = {
        "True Positives  (predicted=1, actual=1)": [],
        "True Negatives  (predicted=0, actual=0)": [],
        "False Positives (predicted=1, actual=0)": [],
        "False Negatives (predicted=0, actual=1)": [],
    }
    for row, p, l in zip(dev_rows, preds, labels):
        if   p == 1 and l == 1: quadrants["True Positives  (predicted=1, actual=1)"].append(row)
        elif p == 0 and l == 0: quadrants["True Negatives  (predicted=0, actual=0)"].append(row)
        elif p == 1 and l == 0: quadrants["False Positives (predicted=1, actual=0)"].append(row)
        elif p == 0 and l == 1: quadrants["False Negatives (predicted=0, actual=1)"].append(row)

    for label, rows in quadrants.items():
        print(f"\n── {label} ({len(rows)} total, showing {min(n, len(rows))}) ──")
        for r in rows[:n]:
            print(f"  [{r['id']}] {r['text'][:140]!r}")


for task, label in [("Opinion", "opinion_label"), ("Misinformation", "misinformation_label")]:
    task_keys = [k for k in metrics_cache if k.endswith(f"— {label}")]
    best_key  = max(task_keys, key=rank_key)
    preds, labels_list, probs = results[best_key]
    m = metrics_cache[best_key]

    print(f"\n{'='*60}")
    print(f"{task} — best model: {best_key.replace(f' — {label}', '')}")
    print(f"Macro F1: {m['macro_f1']:.4f}  |  AUC-ROC: {m.get('auc_roc', float('nan')):.4f}")
    print(f"{'='*60}")
    print_confusion_matrix(preds, labels_list)
    print_quadrant_examples(dev_rows, preds, labels_list, n=8)


Opinion — best model: Lean Motivated Ensemble
Macro F1: 0.7613  |  AUC-ROC: 0.8031
Confusion matrix (rows=true, cols=predicted):
                 pred=0  pred=1
  true=0 (not-op):    89      28
  true=1 (opinion):   19      64

── True Positives  (predicted=1, actual=1) (64 total, showing 8) ──
  [3] 'im praying for all of my friends down in the Caribbean who have no where else to go and are forced to ride through the hurricane. be strong '
  [7] 'The aftermath of a hurricane is horrific. The heat/humidity is excruciating, no water/ice, no bathing, complete darkness, bugs, no warm food'
  [10] "@john19071969 It's unwise to flood the food supply and environment with new plant varieties. Good science requires more prudence."
  [73] 'Irony just died a thousand deaths! ???? http://t.co/dBU30ObDxz'
  [121] 'Freshman: I wish Hurricane Dorian would come our way.\n\nMe, a senior: we had Harvey two years ago.\n\nFish: but no school\n\nMe: we flooded for d'
  [131] 'As you know that Covid 19 ha